In [0]:
%run ../config/config

In [0]:
from pyspark.sql import functions as F

# Ensure Silver Table Exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_catalog}.orders_clean (
    order_id INT NOT NULL,
    branch_id STRING,
    order_date DATE,
    customer_id INT,
    customer_name STRING,
    total_basket DOUBLE
)
USING DELTA
PARTITIONED BY (order_date)
""")


# Read Bronze table
df = spark.read.table(f"{bronze_catalog}.orders")

# Silver Transformation
df = (
    df
    # Change date column to a date datatype
    .withColumn("date", F.col("date").cast("date"))

    # Rename columns
    .withColumnRenamed("date", "order_date")
    .withColumnRenamed("user_id", "customer_id")
    .withColumnRenamed("name_surname", "customer_name")

    # Standardize total_basket
    .withColumn(
        "total_basket",
        F.regexp_replace(F.col("total_basket"), ",", ".").cast("double")
    )
    .withColumn(
        "total_basket",
        F.round(F.col("total_basket"), 2)
    )
)


# Final Schema
silver_df = (
    df.select(
        "order_id",
        "branch_id", 
        "order_date", 
        "customer_id", 
        "customer_name", 
        "total_basket"
    )
)


# Merge into silver table
silver_df.createOrReplaceTempView("staging_orders")

spark.sql(f"""
MERGE INTO {silver_catalog}.orders_clean t
USING staging_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND (
    t.branch_id      <> s.branch_id OR
    t.order_date     <> s.order_date OR
    t.customer_id    <> s.customer_id OR
    t.customer_name  <> s.customer_name OR
    t.total_basket   <> s.total_basket
)
THEN UPDATE SET
  t.branch_id      = s.branch_id,
  t.order_date     = s.order_date,
  t.customer_id    = s.customer_id,
  t.customer_name  = s.customer_name,
  t.total_basket   = s.total_basket

WHEN NOT MATCHED THEN
INSERT (
  order_id,
  branch_id,
  order_date,
  customer_id,
  customer_name,
  total_basket
)
VALUES (
  s.order_id,
  s.branch_id,
  s.order_date,
  s.customer_id,
  s.customer_name,
  s.total_basket
)
""")


# Validate that the Silver table was written successfully
count = spark.table(f"{silver_catalog}.orders_clean").count()

# Ensure table is not empty
assert count > 0, "orders_clean table is empty"

# Log result to job output
print(f"orders_clean validation passed: {count:,} records")